# EHM Synthetic Data Generator — v2
### Phase 1 Rebuild

**What's new in v2 based on your Phase 2 findings:**

1. **Fleet age diversity** — engines span young, mid-life, late-life, and end-of-life so the RAG breakdown gives green, amber, and red
2. **All sensors have dropouts** — every sensor has its own realistic dropout rate, not just three
3. **Diverse sensor faults** — 15% of engines are flagged as having a faulty sensor with elevated outlier rates, plus occasional stuck-sensor events

Run cells top to bottom. The final cell saves `ehm_synthetic_fleet.csv` which you upload to Phase 2.

---

## Step 1 — Setup and Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

plt.style.use('dark_background')
plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor']   = '#1a1d2e'
plt.rcParams['grid.alpha']       = 0.3

RED   = '#ef5350'
AMBER = '#ffa726'
GREEN = '#66bb6a'
BLUE  = '#42a5f5'

print('Imports loaded')

In [ ]:
# Engine configuration — all the constants in one place

ENGINE_CONFIG = {
    'baseline': {
        'EGT':          620.0,
        'N1':            89.0,
        'N2':            91.0,
        'fuel_flow':   2350.0,
        'vibration_1':    0.35,
        'vibration_2':    0.28,
        'oil_temp':      88.0,
        'oil_pressure':  62.0,
        'oil_consumption': 0.08,
        'EPR':            1.52,
    },
    'EGT_redline':      1020.0,
    'EGT_takeoff_new':   940.0,
    'EGT_margin_limit':   15.0,
    'influence_coefficients': {
        'HPT_efficiency':  12.0,
        'HPC_efficiency':   8.0,
        'LPT_efficiency':   4.5,
        'fan_efficiency':   2.0,
    },
    'degradation_rates': {
        'HPT_mean': 0.00009, 'HPT_std': 0.00003,
        'HPC_mean': 0.00007, 'HPC_std': 0.000025,
        'LPT_mean': 0.00004, 'LPT_std': 0.000015,
        'fan_mean': 0.00003, 'fan_std': 0.000012,
    },
    'wash_recovery_factor': 0.70,
    'wash_interval_cycles': 1500,
    'wash_interval_noise':   200,
    'sensor_noise': {
        'EGT': 2.5, 'N1': 0.08, 'N2': 0.06, 'fuel_flow': 12.0,
        'vibration_1': 0.03, 'vibration_2': 0.025,
        'oil_temp': 1.2, 'oil_pressure': 0.8,
        'oil_consumption': 0.005, 'EPR': 0.003,
    },
    # NEW v2: per-sensor dropout rates — every sensor now has one
    'sensor_dropout_rates': {
        'vibration_1':     0.020,   # 2.0%
        'vibration_2':     0.020,   # 2.0%
        'oil_consumption': 0.020,   # 2.0%
        'oil_temp':        0.008,   # 0.8%
        'oil_pressure':    0.008,   # 0.8%
        'fuel_flow':       0.005,   # 0.5%
        'EGT':             0.003,   # 0.3%
        'N1':              0.003,   # 0.3%
        'N2':              0.003,   # 0.3%
        'EPR':             0.004,   # 0.4%
    },
    'sensor_drift_per_1000_cycles': {
        'EGT': 1.5, 'N1': 0.03, 'N2': 0.02, 'fuel_flow': 5.0,
    },
    # NEW v2: sensor fault parameters
    'spike_outlier_rate':        0.010,   # 1% baseline spike rate
    'faulty_sensor_multiplier':  5.0,     # faulty engines: 5x spike rate
    'stuck_sensor_probability':  0.002,   # 0.2% chance of stuck event per cycle
    'stuck_duration_cycles':     (3, 10), # stuck lasts 3-10 cycles
    'faulty_engine_fraction':    0.15,    # 15% of engines have faulty sensor
}

# NEW v2: Fleet age distribution — realistic CFM56 ranges (up to ~22k cycles)
FLEET_AGE_DISTRIBUTION = {
    'young':     {'fraction': 0.30, 'initial_age_range': (500,   5000),  'cycles_to_simulate': 2000},
    'mid_life':  {'fraction': 0.35, 'initial_age_range': (5000,  11000), 'cycles_to_simulate': 2000},
    'late_life': {'fraction': 0.20, 'initial_age_range': (11000, 16000), 'cycles_to_simulate': 1500},
    'end_life':  {'fraction': 0.15, 'initial_age_range': (16000, 21000), 'cycles_to_simulate': 1000},
}

print('Engine configuration loaded')
print()
print('Fleet age distribution:')
for bucket, cfg in FLEET_AGE_DISTRIBUTION.items():
    pct = int(cfg['fraction'] * 100)
    start, end = cfg['initial_age_range']
    print(f'  {bucket:<12}: {pct:>3}% of fleet | starts at cycle {start}-{end} | runs for {cfg["cycles_to_simulate"]} cycles')

## Step 2 — ISA Correction Functions

In [ ]:
def compute_isa_conditions(altitude_ft):
    T_sl, P_sl = 288.15, 1013.25
    L, R, g    = 0.0065, 287.05, 9.80665
    tropopause_ft = 36089
    alt_m = altitude_ft * 0.3048

    if altitude_ft <= tropopause_ft:
        T_isa = T_sl - L * alt_m
        P_isa = P_sl * (T_isa / T_sl) ** (g / (L * R))
    else:
        T_tropo = T_sl - L * (tropopause_ft * 0.3048)
        P_tropo = P_sl * (T_tropo / T_sl) ** (g / (L * R))
        delta_alt_m = (altitude_ft - tropopause_ft) * 0.3048
        T_isa = T_tropo
        P_isa = P_tropo * np.exp(-g * delta_alt_m / (R * T_tropo))

    return T_isa, P_isa


def isa_correction_factors(T_ambient_K, P_ambient_hPa):
    theta = T_ambient_K / 288.15
    delta = P_ambient_hPa / 1013.25
    return theta, delta


print('ISA functions defined')

## Step 3 — Flight Profile Generator

In [ ]:
ROUTE_TYPES = {
    'short_haul':  {'cruise_alt_ft': (28000, 34000), 'cruise_mach': (0.76, 0.80),
                    'flight_hours': (0.8, 2.5),  'oat_deviation': (-15, 15)},
    'medium_haul': {'cruise_alt_ft': (33000, 39000), 'cruise_mach': (0.78, 0.82),
                    'flight_hours': (2.5, 5.5),  'oat_deviation': (-20, 20)},
    'long_haul':   {'cruise_alt_ft': (35000, 41000), 'cruise_mach': (0.80, 0.85),
                    'flight_hours': (5.5, 14.0), 'oat_deviation': (-25, 25)},
}


def generate_flight_profile(route_type, rng):
    route = ROUTE_TYPES[route_type]
    cruise_alt  = rng.uniform(*route['cruise_alt_ft'])
    cruise_mach = rng.uniform(*route['cruise_mach'])
    flight_hrs  = rng.uniform(*route['flight_hours'])
    oat_dev     = rng.uniform(*route['oat_deviation'])

    T_isa, P_isa = compute_isa_conditions(cruise_alt)
    T_actual = T_isa + oat_dev

    thrust_pct = 85.0 + (cruise_alt / 40000) * 3.0 + (oat_dev / 40) * 2.0
    thrust_pct = np.clip(thrust_pct + rng.normal(0, 0.5), 80, 95)

    return {
        'route_type':    route_type,
        'cruise_alt_ft': cruise_alt,
        'cruise_mach':   cruise_mach,
        'flight_hours':  flight_hrs,
        'oat_deviation': oat_dev,
        'T_ambient_K':   T_actual,
        'T_ambient_C':   T_actual - 273.15,
        'P_ambient_hPa': P_isa,
        'thrust_pct':    thrust_pct,
    }


print('Flight profile generator defined')

## Step 4 — Baseline Parameter Models

In [ ]:
def compute_baseline_parameters(flight, config):
    B    = config['baseline']
    T_K  = flight['T_ambient_K']
    P_hPa= flight['P_ambient_hPa']
    thr  = flight['thrust_pct']
    alt  = flight['cruise_alt_ft']

    theta, delta = isa_correction_factors(T_K, P_hPa)

    N1_corr = B['N1'] * (thr / 85.0) ** 0.85
    N2_corr = B['N2'] * (thr / 85.0) ** 0.75
    N1 = N1_corr * np.sqrt(theta)
    N2 = N2_corr * np.sqrt(theta)

    EGT_corr = B['EGT'] * (thr / 85.0) ** 1.2
    # ISA correction uses Kelvin: convert to K, apply theta, convert back to C
    EGT_K = (EGT_corr + 273.15) * theta
    EGT = EGT_K - 273.15

    FF_corr  = B['fuel_flow'] * (thr / 85.0) ** 1.1
    fuel_flow = FF_corr * delta * np.sqrt(theta)

    EPR = B['EPR'] * (thr / 85.0) ** 0.9

    oil_temp        = B['oil_temp'] + (thr - 85.0) * 0.3 + (alt / 40000) * (-5.0)
    oil_pressure    = B['oil_pressure'] * (1.0 - (alt / 40000) * 0.05)
    oil_consumption = B['oil_consumption']

    vibration_1 = B['vibration_1'] * (N1_corr / B['N1']) ** 1.5
    vibration_2 = B['vibration_2'] * (N2_corr / B['N2']) ** 1.5

    return {
        'EGT':           EGT,
        'EGT_corrected': (EGT + 273.15) / theta - 273.15,
        'N1':            N1,
        'N1_corrected':  N1_corr,
        'N2':            N2,
        'N2_corrected':  N2_corr,
        'fuel_flow':     fuel_flow,
        'fuel_flow_corrected': FF_corr,
        'EPR':           EPR,
        'oil_temp':      oil_temp,
        'oil_pressure':  oil_pressure,
        'oil_consumption': oil_consumption,
        'vibration_1':   vibration_1,
        'vibration_2':   vibration_2,
        'theta':         theta,
        'delta':         delta,
    }


print('Baseline parameter model defined')

## Step 5 — Degradation Engine

**v2 change:** engines initialise with a specific age bucket so the fleet has young/mid/old engines.

In [ ]:
def initialise_engine(engine_id, config, route_type, age_bucket, has_faulty_sensor, rng):
    DR = config['degradation_rates']
    age_cfg = FLEET_AGE_DISTRIBUTION[age_bucket]

    HPT_rate = max(0.00002, rng.normal(DR['HPT_mean'], DR['HPT_std']))
    HPC_rate = max(0.00002, rng.normal(DR['HPC_mean'], DR['HPC_std']))
    LPT_rate = max(0.00001, rng.normal(DR['LPT_mean'], DR['LPT_std']))
    fan_rate = max(0.00001, rng.normal(DR['fan_mean'], DR['fan_std']))

    initial_age = int(rng.integers(*age_cfg['initial_age_range']))

    wash_interval = int(rng.normal(config['wash_interval_cycles'], config['wash_interval_noise']))
    next_wash_cycle = initial_age + wash_interval

    faulty_sensor = None
    if has_faulty_sensor:
        faulty_sensor = rng.choice(['EGT', 'vibration_1', 'vibration_2'])

    return {
        'engine_id':         f'ENG-{engine_id:03d}',
        'route_type':        route_type,
        'age_bucket':        age_bucket,
        'HPT_rate':          HPT_rate,
        'HPC_rate':          HPC_rate,
        'LPT_rate':          LPT_rate,
        'fan_rate':          fan_rate,
        'initial_age':       initial_age,
        'cycles_to_simulate': age_cfg['cycles_to_simulate'],
        'next_wash_cycle':   next_wash_cycle,
        'HPT_deg':           HPT_rate * initial_age,
        'HPC_deg':           HPC_rate * initial_age * 0.3,
        'LPT_deg':           LPT_rate * initial_age,
        'fan_deg':           fan_rate * initial_age,
        'HPC_deg_washable':  HPC_rate * initial_age * 0.7,
        'total_cycles':      initial_age,
        'has_faulty_sensor': has_faulty_sensor,
        'faulty_sensor':     faulty_sensor,
        'stuck_sensor_until': -1,
        'stuck_values':      {},
    }


def apply_degradation_step(engine, cycle, config, rng):
    DR = config['degradation_rates']
    event = None

    engine['HPT_deg'] += max(0, rng.normal(engine['HPT_rate'], DR['HPT_std'] * 0.5))
    engine['LPT_deg'] += max(0, rng.normal(engine['LPT_rate'], DR['LPT_std'] * 0.5))
    engine['fan_deg'] += max(0, rng.normal(engine['fan_rate'], DR['fan_std'] * 0.5))

    hpc_step = max(0, rng.normal(engine['HPC_rate'], DR['HPC_std'] * 0.5))
    engine['HPC_deg']          += hpc_step * 0.30
    engine['HPC_deg_washable'] += hpc_step * 0.70

    if cycle >= engine['next_wash_cycle']:
        recovery = engine['HPC_deg_washable'] * config['wash_recovery_factor']
        engine['HPC_deg_washable'] -= recovery
        engine['next_wash_cycle'] = cycle + int(rng.normal(
            config['wash_interval_cycles'], config['wash_interval_noise']))
        event = 'wash'

    if rng.random() < 0.003:
        step = rng.uniform(0.05, 0.25)
        engine['HPT_deg'] += step
        event = f'step_change:{step:.3f}'

    engine['total_cycles'] = cycle
    return engine, event


def compute_degradation_offsets(engine, config):
    IC = config['influence_coefficients']
    total_HPC_deg = engine['HPC_deg'] + engine['HPC_deg_washable']

    delta_EGT = (
        engine['HPT_deg'] * IC['HPT_efficiency'] +
        total_HPC_deg    * IC['HPC_efficiency'] +
        engine['LPT_deg'] * IC['LPT_efficiency'] +
        engine['fan_deg'] * IC['fan_efficiency']
    )
    delta_SFC_pct = (engine['HPT_deg'] * 0.8 + total_HPC_deg * 1.2 + engine['LPT_deg'] * 0.4)
    delta_vib_1 = engine['fan_deg'] * 0.15
    delta_vib_2 = engine['HPT_deg'] * 0.08 + engine['LPT_deg'] * 0.05
    delta_oil_consumption = (engine['HPT_deg'] + engine['LPT_deg']) * 0.003
    delta_N2 = -engine['HPT_deg'] * 0.05 - total_HPC_deg * 0.03

    return {
        'delta_EGT':             delta_EGT,
        'delta_SFC_pct':         delta_SFC_pct,
        'delta_vib_1':           delta_vib_1,
        'delta_vib_2':           delta_vib_2,
        'delta_oil_consumption': delta_oil_consumption,
        'delta_N2':              delta_N2,
    }


print('Degradation engine defined')

## Step 6 — Sensor Noise & Faults (v2 expanded)

Every sensor now has its own dropout rate. Some engines have a faulty sensor with elevated spike rates. Occasional stuck-sensor events freeze a reading for several cycles.

In [ ]:
def add_sensor_noise(params, engine, config, cycle, rng):
    noise         = config['sensor_noise']
    drift_cfg     = config['sensor_drift_per_1000_cycles']
    dropout_rates = config['sensor_dropout_rates']
    noisy = params.copy()

    # Layer 1: Gaussian measurement noise
    for param, sigma in noise.items():
        if param in noisy:
            noisy[param] = noisy[param] + rng.normal(0, sigma)

    # Layer 2: Sensor drift
    drift_factor = cycle / 1000.0
    noisy['EGT']       += drift_cfg['EGT']       * drift_factor * rng.normal(0, 0.1)
    noisy['N1']        += drift_cfg['N1']        * drift_factor * rng.normal(0, 0.1)
    noisy['N2']        += drift_cfg['N2']        * drift_factor * rng.normal(0, 0.1)
    noisy['fuel_flow'] += drift_cfg['fuel_flow'] * drift_factor * rng.normal(0, 0.1)

    # Layer 3: Stuck sensor — if currently in a stuck period, use frozen value
    if cycle <= engine['stuck_sensor_until']:
        for sensor, frozen_val in engine['stuck_values'].items():
            if sensor in noisy:
                noisy[sensor] = frozen_val
    elif rng.random() < config['stuck_sensor_probability']:
        stuck_sensor = rng.choice(['EGT', 'vibration_1', 'oil_pressure'])
        stuck_dur = int(rng.integers(*config['stuck_duration_cycles']))
        engine['stuck_sensor_until'] = cycle + stuck_dur
        engine['stuck_values'] = {stuck_sensor: noisy[stuck_sensor]}

    # Layer 4: Spike outliers — faulty engines get elevated rate on their bad sensor
    spike_rate = config['spike_outlier_rate']

    if engine['has_faulty_sensor']:
        faulty = engine['faulty_sensor']
        if rng.random() < (spike_rate * config['faulty_sensor_multiplier']):
            if faulty == 'EGT':
                noisy['EGT'] += rng.uniform(25, 60) * rng.choice([-1, 1])
            elif faulty == 'vibration_1':
                noisy['vibration_1'] += rng.uniform(0.4, 1.5)
            elif faulty == 'vibration_2':
                noisy['vibration_2'] += rng.uniform(0.3, 1.2)

    # Baseline spikes on any engine
    if rng.random() < spike_rate:
        noisy['EGT'] += rng.uniform(15, 40) * rng.choice([-1, 1])
    if rng.random() < spike_rate:
        noisy['vibration_1'] += rng.uniform(0.3, 1.0)

    # Layer 5: Random dropouts per sensor (every sensor has its own rate)
    for sensor, rate in dropout_rates.items():
        if sensor in noisy and rng.random() < rate:
            noisy[sensor] = np.nan

    return noisy


print('Sensor noise model defined (v2)')
print('  - All 10 sensors have individual dropout rates')
print('  - Stuck-sensor events modelled')
print('  - Engine-level faulty sensor flag')

## Step 7 — Fleet Assembly

In [ ]:
def compute_RUL(engine, config):
    offsets = compute_degradation_offsets(engine, config)
    EGT_takeoff = config['EGT_takeoff_new'] + offsets['delta_EGT'] * 1.1
    current_margin = config['EGT_redline'] - EGT_takeoff

    IC = config['influence_coefficients']
    EGT_rate = (
        engine['HPT_rate'] * IC['HPT_efficiency'] +
        engine['HPC_rate'] * IC['HPC_efficiency'] * 0.30 +
        engine['LPT_rate'] * IC['LPT_efficiency'] +
        engine['fan_rate'] * IC['fan_efficiency']
    ) * 1.1

    margin_to_limit = current_margin - config['EGT_margin_limit']
    if EGT_rate <= 0 or margin_to_limit <= 0:
        return 0
    return max(0, int(margin_to_limit / EGT_rate))


def assign_age_buckets(n_engines, rng):
    buckets = []
    for bucket, cfg in FLEET_AGE_DISTRIBUTION.items():
        n = int(round(n_engines * cfg['fraction']))
        buckets.extend([bucket] * n)
    while len(buckets) < n_engines:
        buckets.append('mid_life')
    buckets = buckets[:n_engines]
    rng.shuffle(buckets)
    return buckets


def assign_faulty_engines(n_engines, config, rng):
    n_faulty = max(1, int(round(n_engines * config['faulty_engine_fraction'])))
    flags = [True] * n_faulty + [False] * (n_engines - n_faulty)
    rng.shuffle(flags)
    return flags


def safe_round(val, n):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return np.nan
    return round(val, n)


def generate_fleet(n_engines=20, seed=42):
    route_mix = {'short_haul': 0.35, 'medium_haul': 0.45, 'long_haul': 0.20}
    rng = np.random.default_rng(seed)
    routes = list(route_mix.keys())
    probs  = list(route_mix.values())

    age_buckets  = assign_age_buckets(n_engines, rng)
    faulty_flags = assign_faulty_engines(n_engines, ENGINE_CONFIG, rng)

    print(f'Generating fleet: {n_engines} engines')
    print('  Age bucket distribution:')
    for b in FLEET_AGE_DISTRIBUTION:
        print(f'    {b:<12}: {age_buckets.count(b)} engines')
    print(f'  Engines with faulty sensors: {sum(faulty_flags)}')
    print()

    records = []
    for eng_idx in range(n_engines):
        route      = rng.choice(routes, p=probs)
        age_bucket = age_buckets[eng_idx]
        faulty     = faulty_flags[eng_idx]

        engine = initialise_engine(eng_idx + 1, ENGINE_CONFIG, route, age_bucket, faulty, rng)
        start_cycle = engine['initial_age']
        end_cycle   = start_cycle + engine['cycles_to_simulate']

        for cycle in range(start_cycle, end_cycle):
            engine, maint_event = apply_degradation_step(engine, cycle, ENGINE_CONFIG, rng)
            flight = generate_flight_profile(route, rng)
            params = compute_baseline_parameters(flight, ENGINE_CONFIG)

            offsets = compute_degradation_offsets(engine, ENGINE_CONFIG)
            params['EGT']              += offsets['delta_EGT']
            # Recompute EGT_corrected properly in Kelvin after adding degradation offset
            params['EGT_corrected']    = (params['EGT'] + 273.15) / params['theta'] - 273.15
            params['fuel_flow']        *= (1 + offsets['delta_SFC_pct'] / 100)
            params['fuel_flow_corrected'] *= (1 + offsets['delta_SFC_pct'] / 100)
            params['vibration_1']      += offsets['delta_vib_1']
            params['vibration_2']      += offsets['delta_vib_2']
            params['oil_consumption']  += offsets['delta_oil_consumption']
            params['N2']               += offsets['delta_N2']
            params['N2_corrected']     += offsets['delta_N2'] / np.sqrt(params['theta'])

            params = add_sensor_noise(params, engine, ENGINE_CONFIG, cycle, rng)

            # After sensor noise, recompute corrected values from noisy raw values
            # so the ISA correction relationship stays mathematically consistent.
            # Skip if the raw value is NaN (dropout).
            if not (isinstance(params.get('EGT'), float) and np.isnan(params['EGT'])):
                params['EGT_corrected'] = (params['EGT'] + 273.15) / params['theta'] - 273.15
            if not (isinstance(params.get('N1'), float) and np.isnan(params['N1'])):
                params['N1_corrected'] = params['N1'] / np.sqrt(params['theta'])
            if not (isinstance(params.get('N2'), float) and np.isnan(params['N2'])):
                params['N2_corrected'] = params['N2'] / np.sqrt(params['theta'])
            if not (isinstance(params.get('fuel_flow'), float) and np.isnan(params['fuel_flow'])):
                params['fuel_flow_corrected'] = params['fuel_flow'] / (params['delta'] * np.sqrt(params['theta']))

            rul = compute_RUL(engine, ENGINE_CONFIG)
            EGT_takeoff = ENGINE_CONFIG['EGT_takeoff_new'] + offsets['delta_EGT'] * 1.1
            EGT_margin = ENGINE_CONFIG['EGT_redline'] - EGT_takeoff

            record = {
                'engine_id':            engine['engine_id'],
                'cycle':                cycle,
                'route_type':           route,
                'age_bucket':           age_bucket,
                'has_faulty_sensor':    faulty,
                'faulty_sensor':        engine['faulty_sensor'] if faulty else 'none',
                'cruise_alt_ft':        round(flight['cruise_alt_ft'], 0),
                'cruise_mach':          round(flight['cruise_mach'], 3),
                'OAT_C':                round(flight['T_ambient_C'], 1),
                'OAT_deviation':        round(flight['oat_deviation'], 1),
                'thrust_pct':           round(flight['thrust_pct'], 2),
                'theta':                round(params['theta'], 4),
                'delta':                round(params['delta'], 4),
                'EGT':                  safe_round(params.get('EGT'), 1),
                'N1':                   safe_round(params.get('N1'), 2),
                'N2':                   safe_round(params.get('N2'), 2),
                'fuel_flow':            safe_round(params.get('fuel_flow'), 1),
                'EPR':                  safe_round(params.get('EPR'), 3),
                'vibration_1':          safe_round(params.get('vibration_1'), 3),
                'vibration_2':          safe_round(params.get('vibration_2'), 3),
                'oil_temp':             safe_round(params.get('oil_temp'), 1),
                'oil_pressure':         safe_round(params.get('oil_pressure'), 1),
                'oil_consumption':      safe_round(params.get('oil_consumption'), 4),
                'EGT_corrected':        safe_round(params.get('EGT_corrected'), 1),
                'N1_corrected':         safe_round(params.get('N1_corrected'), 2),
                'N2_corrected':         safe_round(params.get('N2_corrected'), 2),
                'fuel_flow_corrected':  safe_round(params.get('fuel_flow_corrected'), 1),
                'EGT_margin':           round(EGT_margin, 1),
                'HPT_degradation':      round(engine['HPT_deg'], 4),
                'HPC_degradation':      round(engine['HPC_deg'] + engine['HPC_deg_washable'], 4),
                'LPT_degradation':      round(engine['LPT_deg'], 4),
                'fan_degradation':      round(engine['fan_deg'], 4),
                'SFC_degradation_pct':  round(offsets['delta_SFC_pct'], 3),
                'RUL':                  rul,
                'maintenance_event':    maint_event if maint_event else 'none',
                'is_anomaly':           1 if (maint_event and 'step_change' in str(maint_event)) else 0,
            }
            records.append(record)

        if (eng_idx + 1) % 5 == 0:
            print(f'  Engine {eng_idx + 1}/{n_engines} complete')

    df = pd.DataFrame(records)
    print(f'\nFleet generation complete: {df.shape[0]:,} rows, {df.shape[1]} columns')
    return df


df = generate_fleet(n_engines=20, seed=42)

## Step 8 — Validation Plots

Check that v2 actually produced what we intended — mixed fleet, per-sensor dropouts, faulty engines.

In [ ]:
# Validation 1: Fleet age / RUL distribution
latest = df.sort_values('cycle').groupby('engine_id').last().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(latest['RUL'], bins=15, color=BLUE, edgecolor='#0f1117', alpha=0.85)
axes[0].axvline(latest['RUL'].median(), color=AMBER, linestyle='--', linewidth=1.5,
                label=f'Median RUL: {int(latest["RUL"].median())}')
axes[0].set_xlabel('Remaining Useful Life (cycles)')
axes[0].set_ylabel('Number of Engines')
axes[0].set_title('v2: Fleet RUL Distribution\n(Should span young to end-of-life)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

margins_sorted = latest.sort_values('EGT_margin')['EGT_margin'].values
bar_colors = [RED if m < 30 else AMBER if m < 50 else GREEN for m in margins_sorted]

axes[1].bar(range(len(latest)), margins_sorted, color=bar_colors, edgecolor='#0f1117')
axes[1].axhline(30, color=RED,   linestyle='--', linewidth=1.5, label='Red (30°C)')
axes[1].axhline(50, color=AMBER, linestyle='--', linewidth=1.5, label='Amber (50°C)')
axes[1].set_xlabel('Engine (sorted by margin)')
axes[1].set_ylabel('Current EGT Margin (°C)')
axes[1].set_title('v2: Current EGT Margin per Engine\n(Should span red / amber / green)')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

n_red   = ((latest['EGT_margin'] < 30) | (latest['RUL'] < 200)).sum()
n_amber = (((latest['EGT_margin'] < 50) & (latest['EGT_margin'] >= 30)) |
           ((latest['RUL'] < 500) & (latest['RUL'] >= 200))).sum()
n_green = len(latest) - n_red - n_amber
print('\nExpected RAG breakdown at end of simulation:')
print(f'  RED   engines: {n_red}')
print(f'  AMBER engines: {n_amber}')
print(f'  GREEN engines: {n_green}')

In [ ]:
# Validation 2: Dropout rates per sensor (should show all sensors)
sensor_cols = ['vibration_1', 'vibration_2', 'oil_consumption',
               'EGT', 'N1', 'N2', 'fuel_flow', 'oil_temp', 'oil_pressure', 'EPR']

dropout_pcts = [df[c].isna().mean() * 100 for c in sensor_cols]
dropout_df = pd.DataFrame({'sensor': sensor_cols, 'dropout_pct': dropout_pcts}).sort_values('dropout_pct')
colors_do = [RED if p > 3 else AMBER if p > 1 else GREEN for p in dropout_df['dropout_pct']]

fig, ax = plt.subplots(figsize=(11, 5))
ax.barh(dropout_df['sensor'], dropout_df['dropout_pct'], color=colors_do, edgecolor='#0f1117')
ax.axvline(1, color=AMBER, linestyle='--', linewidth=1.5, label='1% watch')
ax.axvline(3, color=RED,   linestyle='--', linewidth=1.5, label='3% action')
ax.set_xlabel('Dropout Rate (%)')
ax.set_title('v2: Per-Sensor Dropout Rates (All 10 sensors covered)')
ax.legend()
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print('Dropout rates:')
for _, row in dropout_df.iterrows():
    print(f'  {row["sensor"]:<18}: {row["dropout_pct"]:.2f}%')

In [ ]:
# Validation 3: Outlier rates per engine (faulty engines should show elevated rates)
outlier_rates = []

for engine_id in df['engine_id'].unique():
    eng_df = df[df['engine_id'] == engine_id]
    egt = eng_df['EGT']
    roll_mean = egt.rolling(50).mean()
    roll_std  = egt.rolling(50).std()
    z = (egt - roll_mean).abs() / (roll_std + 1e-9)
    pct = (z > 3).sum() / len(egt) * 100
    is_faulty = eng_df['has_faulty_sensor'].iloc[0]
    faulty_on = eng_df['faulty_sensor'].iloc[0]
    outlier_rates.append({
        'engine_id':   engine_id,
        'outlier_pct': pct,
        'is_faulty':   is_faulty,
        'faulty_on':   faulty_on,
    })

or_df = pd.DataFrame(outlier_rates).sort_values('outlier_pct')
colors_o = [RED if f else AMBER if p > 0.5 else GREEN
            for f, p in zip(or_df['is_faulty'], or_df['outlier_pct'])]

fig, ax = plt.subplots(figsize=(11, 7))
ax.barh(or_df['engine_id'].str.replace('ENG-',''), or_df['outlier_pct'],
        color=colors_o, edgecolor='#0f1117')
ax.axvline(0.5, color=AMBER, linestyle='--', linewidth=1.5, label='0.5% watch')
ax.axvline(1.0, color=RED,   linestyle='--', linewidth=1.5, label='1% suspect')
ax.set_xlabel('EGT Outlier Rate (%)')
ax.set_ylabel('Engine')
ax.set_title('v2: Outlier Rates per Engine\n(Red bars = engines flagged as faulty in ground-truth)')
ax.legend()
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print('Engines with faulty sensors (ground truth):')
for _, row in or_df[or_df['is_faulty']].iterrows():
    print(f'  {row["engine_id"]}: faulty sensor = {row["faulty_on"]}, EGT outlier rate = {row["outlier_pct"]:.2f}%')

## Step 9 — Export the CSV

This creates `ehm_synthetic_fleet.csv` in your Colab session. Use the file browser (folder icon on the left of Colab) to download it, then upload it to your Phase 2 notebook.

In [ ]:
df.to_csv('ehm_synthetic_fleet.csv', index=False)
print('Saved: ehm_synthetic_fleet.csv')
print(f'Rows: {len(df):,}')
print(f'Columns: {df.shape[1]}')
print()
print('To download the file:')
print('  1. Open the file browser on the left side of Colab (folder icon)')
print('  2. Right-click on ehm_synthetic_fleet.csv')
print('  3. Click Download')
print()
print('Then upload it to Section 1 of your Phase 2 notebook.')

In [ ]:
# Auto-download the CSV if running in Colab
try:
    from google.colab import files
    files.download('ehm_synthetic_fleet.csv')
    print('Download started')
except ImportError:
    print('Not running in Colab — file is saved in the working directory')